# TerraWatch B: RSCaMa retrained under the B protocol (Colab or Kaggle GPU)

**What this does.** Retrains RSCaMa from the public CLIP ViT-B/32 weights with the upstream recipe on a seeded 20% of the LEVIR-MCI train split, picks the checkpoint on our 50 development pairs, then captions the **same 50 development + 100 screening pairs** used for every B0 system. The screening pairs are never loaded before that final pass, so they are genuinely unseen (unlike the released checkpoint, which was selected on val). It also runs the released checkpoint on CUDA for the parity check.

**Data.**
- LEVIR-MCI, Hugging Face copy by the authors: https://huggingface.co/datasets/lcybuaa/LEVIR-MCI (revision `dff09c7`)
- Dataset project page: https://github.com/Chen-Yang-Liu/LEVIR-MCI · captions from LEVIR-CC: https://github.com/Chen-Yang-Liu/RSICC
- Used here: 1,313 train pairs for training + 50 train pairs for checkpoint selection (1,363 = 20.0% of 6,815 train) + the 100 val screening pairs. Test split is not downloaded. Only these pairs are fetched (about 400 MB of the 2.8 GB archive, by HTTP range requests).
- Academic use only: keep the runtime and any copies private.

**Model.** RSCaMa, https://github.com/Chen-Yang-Liu/RSCaMa (commit `aa68e4c`); released checkpoint https://huggingface.co/lcybuaa/RSCaMa for the parity run only.

**How to run.** Needs an NVIDIA GPU (a TPU will not work). Training takes roughly 1-2 hours.
- *Colab:* Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Drive is mounted so training resumes after a disconnect: run all again.
- *Kaggle:* Create > New Notebook > File > Import Notebook (this file). In the right sidebar: Session options > Accelerator > GPU T4 x2, and Internet on (needs a phone-verified account). Then Run All. Kaggle sessions stop after about 12 hours idle or total; results are in `/kaggle/working`.

The last cell writes `b_rscama_results.zip` (captions and logs, no images or weights); send that file back.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle')
ON_COLAB = not ON_KAGGLE and os.path.exists('/var/colab/hostname')  # Kaggle images also carry Colab env vars
for d in ('/content/b', '/content/cache'):  # same paths on every platform
    os.makedirs(d, exist_ok=True)
import urllib.request
try:
    urllib.request.urlopen('https://huggingface.co', timeout=15)
except Exception as e:
    raise SystemExit('No internet in this session. Kaggle: right sidebar > Session options > Internet on '
                     '(needs a phone-verified account), then run all again. ' + repr(e))
OUT = '/kaggle/working/out' if ON_KAGGLE else '/content/out'
if ON_COLAB:  # Drive keeps last.pth across disconnects; just run all again to resume
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/terrawatch_b_rscama'
os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('colab' if ON_COLAB else 'kaggle' if ON_KAGGLE else 'other', OUT)

In [ ]:
%%bash
set -e
mkdir -p /content/cache /content/b
pip -q install transformers==4.39.3 einops==0.6.1 ftfy regex nltk "setuptools<70" wheel
pip -q install --no-build-isolation "git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
[ -d /content/cache/rscama_src ] || git clone -q https://github.com/Chen-Yang-Liu/RSCaMa /content/cache/rscama_src
git -C /content/cache/rscama_src checkout -q aa68e4c6933c7dd0f2e2f9cbd7642d61929c86a5
git -C /content/cache/rscama_src rev-parse HEAD

## Sources (same files as `POC/execution`, tested locally)

In [ ]:
%%writefile /content/b/remote_zip.py
"""Read members of a large remote zip over HTTP range requests.

Lets us index LEVIR-MCI (2.8 GB) and extract only the pairs in the frozen
manifest, so unselected images are never downloaded or inspected.
Standard library only.
"""
import io
import struct
import time
import urllib.error
import urllib.request
import zipfile
import zlib

LEVIR_MCI_URL = (
    "https://huggingface.co/datasets/lcybuaa/LEVIR-MCI/resolve/"
    "dff09c709999b5b1f1f1806024811c65a6b14dfe/LEVIR-MCI-dataset.zip"
)


class HttpRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP with a small read-ahead cache."""

    def __init__(self, url, block=1 << 20):
        self.url = url
        self.block = block
        self.pos = 0
        self.cache_start, self.cache = 0, b""
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req) as r:
            self.size = int(r.headers["Content-Length"])
            self.final_url = r.url
        self.bytes_fetched = 0

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self.pos = offset
        elif whence == io.SEEK_CUR:
            self.pos += offset
        else:
            self.pos = self.size + offset
        return self.pos

    def _fetch(self, start, end, attempts=6):
        """Range GET; retries 429/5xx and dropped connections with backoff (HF CDN sheds bursts)."""
        req = urllib.request.Request(
            self.final_url, headers={"Range": f"bytes={start}-{end - 1}"}
        )
        for k in range(attempts):
            try:
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                break
            except urllib.error.HTTPError as e:
                if e.code != 429 and e.code < 500 or k == attempts - 1:
                    raise
                if e.headers.get("Retry-After", "").isdigit():  # Zenodo says how long to back off
                    time.sleep(int(e.headers["Retry-After"]))
            except (urllib.error.URLError, TimeoutError, ConnectionError):
                if k == attempts - 1:
                    raise
            time.sleep(2 ** k)
        self.bytes_fetched += len(data)
        return data

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        end = self.pos + n
        c0, c1 = self.cache_start, self.cache_start + len(self.cache)
        if not (c0 <= self.pos and end <= c1):
            fetch_end = min(self.size, max(end, self.pos + self.block))
            self.cache_start, self.cache = self.pos, self._fetch(self.pos, fetch_end)
        off = self.pos - self.cache_start
        out = self.cache[off:off + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)


def open_remote_zip(url=LEVIR_MCI_URL):
    raw = HttpRangeFile(url)
    return zipfile.ZipFile(raw), raw


def fetch_members(zf, raw, names, workers=16):
    """Fetch many members in parallel by exact byte range; CRC-checked.

    One request per member (local header + data) instead of serial seeks.
    """
    from concurrent.futures import ThreadPoolExecutor

    def one(name):
        info = zf.getinfo(name)
        start = info.header_offset
        span = 30 + len(info.orig_filename.encode()) + 1024 + info.compress_size
        buf = raw._fetch(start, min(raw.size, start + span))
        n, e = struct.unpack("<HH", buf[26:30])
        data = buf[30 + n + e: 30 + n + e + info.compress_size]
        if info.compress_type == zipfile.ZIP_DEFLATED:
            data = zlib.decompress(data, -15)
        elif info.compress_type != zipfile.ZIP_STORED:
            raise ValueError(f"unsupported compression {info.compress_type}")
        if zlib.crc32(data) != info.CRC:
            raise ValueError(f"CRC mismatch for {name}")
        return name, data

    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(one, names))


In [ ]:
%%writefile /content/b/common.py
"""Shared helpers: manifest loading, run records, timing."""
import csv
import hashlib
import json
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

EXEC_ROOT = Path(__file__).resolve().parents[1]
# B_DATA=data_ood/<name> points every script at an out-of-domain set (scripts/build_ood.py)
DATA = EXEC_ROOT / os.environ.get("B_DATA", "data")
RUNS = EXEC_ROOT / "runs" / ("ood/" + DATA.name if "B_DATA" in os.environ else "")

PROMPT_V1 = (
    "Image T1 is earlier and T2 is later. Describe visible building or road changes "
    "in at most two sentences. Use image-relative locations. If the evidence is "
    "insufficient, state that. Do not infer legality, ownership or cause. Do not give "
    "an exact count unless individual objects are distinguishable. If no relevant "
    "change is visible, say so."
)
PROMPT_REVISION = "B-prompt-v1 (B_captioning.md, unrevised)"


def load_manifest(roles=None):
    with open(DATA / "manifest.csv") as f:
        rows = list(csv.DictReader(f))
    if roles:
        rows = [r for r in rows if r["poc_role"] in roles]
    if any(r["official_split"] == "test" for r in rows):
        raise SystemExit("Sealed test pairs are not available to runs yet.")
    return rows


def pair_paths(row):
    return DATA / row["t1_path"], DATA / row["t2_path"]


def dataset_meta():
    p = DATA / "manifest.json"
    return json.loads(p.read_text()) if p.exists() else {}


def manifest_sha256():
    return hashlib.sha256((DATA / "manifest.csv").read_bytes()).hexdigest()


def git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=EXEC_ROOT, text=True
        ).strip()
    except Exception:
        return None


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def environment():
    import torch

    return {
        "machine": platform.machine(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "mps": torch.backends.mps.is_available(),
        "cuda": torch.cuda.is_available(),
    }


def new_run_record(**kw):
    template = json.loads(
        (EXEC_ROOT.parent / "benchmarks" / "templates" / "run_record.json").read_text()
    )
    rec = {k: None for k in template}
    rec.update(
        record_type="run_record",
        owner="Sujal Raj Pradhan",
        code_commit=git_commit(),
        track="M",
        dataset_version=dataset_meta().get("dataset_version", "hf:lcybuaa/LEVIR-MCI@dff09c7"),
        manifest_sha256=manifest_sha256(),
        environment_record=environment(),
        status="running",
        start_utc=now(),
        errors=[],
    )
    rec.update(kw)
    return rec


def percentile(xs, q):
    xs = sorted(xs)
    if not xs:
        return None
    k = (len(xs) - 1) * q
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return round(xs[lo] + (xs[hi] - xs[lo]) * (k - lo), 3)


class Timer:
    def __enter__(self):
        self.t = time.perf_counter()
        return self

    def __exit__(self, *a):
        self.s = time.perf_counter() - self.t


In [ ]:
%%writefile /content/b/caption_rscama.py
"""B0 captions from RSCaMa (C02), author checkpoint, on Apple MPS/CPU.

Runs the unmodified upstream code (commit aa68e4c, in cache/rscama_src) with:
  1. a pure-PyTorch shim for the two CUDA kernels it calls
     (selective_scan_fn = mamba_ssm's reference scan, causal_conv1d_fn =
     grouped conv1d + SiLU); weights and state_dict keys are unchanged;
  2. the README-required CLIP patch (VisionTransformer returns patch tokens);
  3. .cuda() redirected to the local device.
Native CLIP preprocessing (224 px), author vocab rebuilt at threshold 5, greedy decoding
(decoder.sample k=1), max_length 42 -- as in the author's test.py.

  .venv-rscama/bin/python scripts/caption_rscama.py --roles development_smoke
"""
import argparse
import json
import os
import sys
import traceback
from pathlib import Path

import torch
import torch.nn.functional as F

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from common import RUNS, EXEC_ROOT, Timer, load_manifest, new_run_record, now, pair_paths, percentile  # noqa: E402

SRC = EXEC_ROOT / "cache" / "rscama_src"
CKPT = EXEC_ROOT / "weights" / "rscama" / "RSCaMa.pth"
UPSTREAM_COMMIT = "aa68e4c6933c7dd0f2e2f9cbd7642d61929c86a5"
CKPT_REVISION = "3e527e6656b62cc615eab1d2f3fb73777cbbf62c"


def selective_scan_ref(u, delta, A, B, C, D=None, z=None, delta_bias=None,
                       delta_softplus=False, return_last_state=False):
    """mamba_ssm.ops.selective_scan_interface.selective_scan_ref (real A, B/C: b n l)."""
    dtype_in = u.dtype
    u, delta = u.float(), delta.float()
    if delta_bias is not None:
        delta = delta + delta_bias[..., None].float()
    if delta_softplus:
        delta = F.softplus(delta)
    B, C = B.float(), C.float()
    x = A.new_zeros((u.shape[0], A.shape[0], A.shape[1]))
    deltaA = torch.exp(torch.einsum("bdl,dn->bdln", delta, A))
    deltaB_u = torch.einsum("bdl,bnl,bdl->bdln", delta, B, u)
    ys = []
    for i in range(u.shape[2]):
        x = deltaA[:, :, i] * x + deltaB_u[:, :, i]
        ys.append(torch.einsum("bdn,bn->bd", x, C[:, :, i]))
    y = torch.stack(ys, dim=2)
    out = y if D is None else y + u * D.float()[:, None]
    if z is not None:
        out = out * F.silu(z.float())
    out = out.to(dtype_in)
    return (out, x) if return_last_state else out


def causal_conv1d_ref(x, weight, bias=None, activation=None):
    L = x.shape[-1]
    y = F.conv1d(x, weight.unsqueeze(1), bias, padding=weight.shape[1] - 1, groups=x.shape[1])[..., :L]
    return F.silu(y) if activation in ("silu", "swish") else y


def build_author_vocab(captions_json, threshold=5):
    """Rebuild the vocabulary with the author's own tokenize/build_vocab (preprocess_data.py).

    The shipped data/LEVIR_CC/vocab.json has 501 tokens (threshold 4); the released
    checkpoint's decoder has 468, which threshold 5 (the README's setting) reproduces.
    Word counts use caption text from all splits, as upstream does; no images.
    """
    import ast
    src = (SRC / "preprocess_data.py").read_text()
    keep = [n for n in ast.parse(src).body
            if (isinstance(n, ast.FunctionDef) and n.name in ("tokenize", "build_vocab"))
            or (isinstance(n, ast.Assign) and any(getattr(t, "id", "") == "SPECIAL_TOKENS" for t in n.targets))]
    ns = {}
    exec(compile(ast.Module(body=keep, type_ignores=[]), "preprocess_data", "exec"), ns)
    data = json.load(open(captions_json))
    seqs = [(img["filename"], [ns["tokenize"](s["raw"], add_start_token=True, add_end_token=True,
                                              punct_to_keep=[";", ","], punct_to_remove=["?", "."])
                               for s in img["sentences"]]) for img in data["images"]]
    return ns["build_vocab"](seqs, threshold)


def install_patches(device, shim=True):
    """shim=False keeps mamba_ssm/causal_conv1d kernels (CUDA parity run)."""
    import transformers.models.mamba.modeling_mamba as mm
    if shim:
        mm.selective_scan_fn = selective_scan_ref
        mm.causal_conv1d_fn = causal_conv1d_ref
    elif mm.selective_scan_fn is None or mm.causal_conv1d_fn is None:
        raise RuntimeError("mamba_ssm / causal_conv1d kernels not importable")
    for name in ("causal_conv1d_update", "selective_state_update", "mamba_inner_fn"):
        if not hasattr(mm, name) or getattr(mm, name) is None:
            setattr(mm, name, None)

    import clip.model as cm

    def vit_forward(self, x):  # README / PromptCC issue #3: also return patch tokens before ln_post
        x = self.conv1(x)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)
        x = x.permute(1, 0, 2)
        x = self.transformer(x)
        x = x.permute(1, 0, 2)
        featuremap = x[:, 1:, :]
        x = self.ln_post(x[:, 0, :])
        if self.proj is not None:
            x = x @ self.proj
        return x, featuremap

    cm.VisionTransformer.forward = vit_forward
    torch.Tensor.cuda = lambda t, *a, **k: t.to(device)
    torch.nn.Module.cuda = lambda m, *a, **k: m.to(device)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--roles", default="development_smoke")
    ap.add_argument("--device", default="mps" if torch.backends.mps.is_available() else "cpu")
    ap.add_argument("--ckpt", default=str(CKPT), help="released RSCaMa.pth, or best.pth from colab/rscama_colab.py")
    args = ap.parse_args()
    args.ckpt = str(Path(args.ckpt).resolve())  # main() chdirs into the upstream source tree
    retrained = Path(args.ckpt).name == "best.pth"
    mid = "C02-r20" if retrained else "C02"
    device = torch.device(args.device)
    install_patches(device)

    sys.path.insert(0, str(SRC))
    os.chdir(SRC)  # upstream modules use relative paths
    import clip
    from PIL import Image
    from model.mamba_block import MambaMixer
    from model.model_decoder import DecoderTransformer
    from model.model_encoder_attMamba import AttentiveEncoder, Encoder

    MambaMixer.forward = lambda self, h, h2, cache_params=None, cache_params_2=None: self.cuda_kernels_forward(h, h2)

    roles = args.roles.split(",")
    rows = load_manifest(roles)
    run_id = f"B0_C02r20_rscama_retrained20_{'+'.join(roles)}" if retrained else f"B0_C02_rscama_{'+'.join(roles)}"
    out = RUNS / run_id
    out.mkdir(parents=True, exist_ok=True)
    rec = new_run_record(
        run_id=run_id, role_model_id=mid,
        model_identity={"repo": "https://github.com/Chen-Yang-Liu/RSCaMa", "upstream_commit": UPSTREAM_COMMIT,
                        "hf_checkpoint": "retrained best.pth (Kaggle v6)" if retrained else "lcybuaa/RSCaMa/RSCaMa.pth", "backbone": "CLIP-ViT-B/32",
                        "kernel_path": "pure-PyTorch reference shim (no mamba_ssm/causal_conv1d CUDA kernels)"},
        checkpoint_revision=None if retrained else CKPT_REVISION,
        training_data_exposure=("Retrained by us on 1,313 LEVIR-MCI train pairs (colab/rscama_colab.py)." if retrained else
                                "Trained on the LEVIR-CC official train split (author paper). Development pairs "
                                "here come from that split (seen in training); screening pairs from val."),
        split="+".join(sorted({r["official_split"] for r in rows})), sample_count=len(rows), condition="B0",
        processor_config="CLIP preprocess: bicubic resize 224, centre crop 224, CLIP mean/std",
        precision_and_quantisation=f"float32, device={args.device}",
        prompt_revision="n/a (specialist captioner, no prompt)", seed=None,
        limitations="Kernel shim not parity-checked against CUDA kernels yet; greedy decoding as upstream test.py.",
    )
    rec["decoding"] = {"method": "decoder.sample(k=1) greedy", "max_length": 42}

    with Timer() as load_t:
        ckpt = torch.load(args.ckpt, map_location="cpu", mmap=not retrained, weights_only=not retrained)
        # the retrained checkpoint carries its own vocabulary (built from its training subset)
        word_vocab = ckpt["vocab"] if retrained else build_author_vocab(
            EXEC_ROOT / "data" / "levir_mci_meta" / "LevirCCcaptions.json", 5)
        assert len(word_vocab) == ckpt["decoder_dict"]["wdc.weight"].shape[0], len(word_vocab)
        _, preprocess = clip.load("ViT-B/32", device="cpu", jit=False)
        encoder = Encoder("CLIP-ViT-B/32")
        encoder_trans = AttentiveEncoder(n_layers=3, feature_size=[7, 7, 768], heads=8, dropout=0.1)
        decoder = DecoderTransformer(decoder_type="transformer_decoder", embed_dim=768,
                                     vocab_size=len(word_vocab), max_lengths=42, word_vocab=word_vocab,
                                     n_head=8, n_layers=1, dropout=0.1)
        load_report = {"vocab": f"{'from checkpoint' if retrained else 'rebuilt with author preprocess_data.py, threshold 5'}, {len(word_vocab)} tokens"}
        for name, mod, key in (("encoder", encoder, "encoder_dict"), ("encoder_trans", encoder_trans, "encoder_trans_dict"),
                               ("decoder", decoder, "decoder_dict")):
            mod.load_state_dict(ckpt[key], strict=True)
            load_report[name] = "strict ok"
            mod.eval().to(device)
        del ckpt
    rec["cold_load_s"] = round(load_t.s, 2)
    rec["load_report"] = load_report
    idx2word = {v: k for k, v in word_vocab.items()}
    skip = {word_vocab["<START>"], word_vocab["<END>"], word_vocab["<NULL>"]}

    times, done = [], 0
    with open(out / "captions.jsonl", "w") as f:
        for i, row in enumerate(rows):
            p1, p2 = pair_paths(row)
            item = {"pair_id": row["pair_id"], "run_id": run_id, "model": mid, "condition": "B0",
                    "poc_role": row["poc_role"]}
            try:
                a = preprocess(Image.open(p1).convert("RGB")).unsqueeze(0).to(device)
                b = preprocess(Image.open(p2).convert("RGB")).unsqueeze(0).to(device)
                with torch.no_grad(), Timer() as t:
                    feat1, feat2 = encoder(a, b)
                    feat = encoder_trans(feat1, feat2)
                    seq = decoder.sample(feat, k=1)
                    if device.type == "mps":
                        torch.mps.synchronize()
                words = [idx2word[w] for w in seq if w not in skip]
                item.update(raw_text=" ".join(words), status="ok", output_tokens=len(words),
                            hit_token_limit=len(seq) >= 43, gen_s=round(t.s, 3))
                if i > 0:
                    times.append(t.s)
                done += 1
            except Exception as e:
                item.update(raw_text=None, status="error", error=repr(e))
                rec["errors"].append({"pair_id": row["pair_id"], "error": traceback.format_exc()[-800:]})
            f.write(json.dumps(item) + "\n")
            f.flush()
            print(f"[{i + 1}/{len(rows)}] {row['pair_id']} {item.get('gen_s')}s :: {item.get('raw_text')}", flush=True)

    rec.update(status="completed" if done == len(rows) else "completed_with_errors", end_utc=now(),
               completion_rate=round(done / len(rows), 4), warm_latency_median_s=percentile(times, 0.5),
               warm_latency_p95_s=percentile(times, 0.95),
               peak_gpu_memory_mb=round(torch.mps.driver_allocated_memory() / 2**20) if device.type == "mps" else None,
               raw_output_path=str((out / "captions.jsonl").relative_to(RUNS.parent)))
    json.dump(rec, open(out / "run_record.json", "w"), indent=1)
    print(json.dumps({k: rec[k] for k in ("run_id", "status", "cold_load_s", "warm_latency_median_s", "peak_gpu_memory_mb", "completion_rate")}))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/b/rscama_colab.py
"""RSCaMa retrained under the B protocol, plus the CUDA parity run (Colab T4 or any CUDA box).

Why retrain instead of fine-tuning the released checkpoint further: that checkpoint
already saw the whole LEVIR-CC train split and was selected on val BLEU-4, so the
screening (val) pairs are not unseen for it. This run starts again from the public
CLIP ViT-B/32 weights with the upstream recipe, trains on a seeded 20% of the train
split, selects the checkpoint on the 50 development pairs (held out of training) and
never loads a val image until the final captioning pass.

  fetch   download only the needed pairs from LEVIR-MCI (HTTP range reads)
  train   upstream recipe: Adam 1e-4, ~64 captions per update (12 pairs x 5), last 6 CLIP blocks tuned
  caption greedy captions for development + screening (our captions.jsonl format)
  parity  released checkpoint on screening, shim or real kernels (--kernels)

Dataset: LEVIR-MCI, https://huggingface.co/datasets/lcybuaa/LEVIR-MCI (rev dff09c7).
"""
import argparse
import hashlib
import json
import os
import random
import sys
import time
from pathlib import Path

import torch

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent / "scripts"))  # local checkout; on Colab all files sit together
from caption_rscama import SRC, build_author_vocab, install_patches  # noqa: E402
from remote_zip import LEVIR_MCI_URL, fetch_members, open_remote_zip  # noqa: E402

ROOT = "LEVIR-MCI-dataset"
SEED = 42
N_DEV, N_SCREEN = 50, 100
CAP = 0.20
NOCHANGE = ["<START>", "the", "scene", "is", "the", "same", "as", "before", "<END>"]
# sha256 of the newline-joined file names, computed from the frozen local manifest (sha c5dde811...)
EXPECTED = {"development": "36959f028fb958c523391cb79b78c0dfa8879314a6fae69c1b4d520d8c6d8176",
            "screening": "ce994d9e4f234dd6e2d0537c453eaff7c131a92b745ee9e05aa5a2bc4822969e",
            "finetune": "0aa6974f154df1c9d225fae71d1cec5c942a4b592b2e83ac933a975aa4456416"}


def ids_sha(recs):
    return hashlib.sha256("\n".join(r["filename"] for r in recs).encode()).hexdigest()


def select(caps):
    """Same rule as scripts/build_manifest.py, plus the fine-tune pool.

    finetune = Random('42-train-finetune').sample(train minus development, n) with
    n chosen so development + finetune = 20% of train (the POC cap).
    """
    pool = {s: sorted([r for r in caps if r["filepath"] == s], key=lambda r: r["filename"]) for s in ("train", "val")}
    dev = random.Random(f"{SEED}-train").sample(pool["train"], N_DEV)
    scr = random.Random(f"{SEED}-val").sample(pool["val"], N_SCREEN)
    dev_names = {r["filename"] for r in dev}
    n_ft = int(CAP * len(pool["train"])) - N_DEV
    rest = [r for r in pool["train"] if r["filename"] not in dev_names]
    ft = random.Random(f"{SEED}-train-finetune").sample(rest, n_ft)
    sel = {"development": dev, "screening": scr, "finetune": ft}
    for k, v in sel.items():
        if EXPECTED[k] is not None:
            assert ids_sha(v) == EXPECTED[k], f"{k} selection differs from the frozen manifest"
    return sel


def cmd_fetch(a):
    out = Path(a.data)
    z, raw = open_remote_zip()
    caps = json.loads(z.read(f"{ROOT}/LevirCCcaptions.json"))["images"]
    (out / "LevirCCcaptions.json").parent.mkdir(parents=True, exist_ok=True)
    (out / "LevirCCcaptions.json").write_text(json.dumps({"images": caps}))
    sel = select(caps)
    wanted = [f"{ROOT}/images/{r['filepath']}/{sub}/{r['filename']}"
              for recs in sel.values() for r in recs for sub in ("A", "B")]
    t = time.time()
    for i in range(0, len(wanted), 400):
        for name, b in fetch_members(z, raw, wanted[i:i + 400]).items():
            dest = out / name[len(ROOT) + 1:]
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(b)
        print(f"{min(i + 400, len(wanted))}/{len(wanted)} files", flush=True)
    info = {"source": LEVIR_MCI_URL, "bytes_fetched": raw.bytes_fetched, "fetch_s": round(time.time() - t, 1),
            **{f"{k}_pairs": len(v) for k, v in sel.items()}, **{f"{k}_ids_sha256": ids_sha(v) for k, v in sel.items()},
            "train_split_pairs": sum(r["filepath"] == "train" for r in caps)}
    json.dump(info, open(out / "fetch.json", "w"), indent=1)
    print(json.dumps(info, indent=1))


def load_upstream(device, shim=True):
    install_patches(device, shim=shim)
    sys.path.insert(0, str(SRC))
    cwd = os.getcwd()
    os.chdir(SRC)
    import clip
    from model.mamba_block import MambaMixer
    from model.model_decoder import DecoderTransformer
    from model.model_encoder_attMamba import AttentiveEncoder, Encoder
    MambaMixer.forward = lambda self, h, h2, cache_params=None, cache_params_2=None: self.cuda_kernels_forward(h, h2)
    _, preprocess = clip.load("ViT-B/32", device="cpu", jit=False)
    os.chdir(cwd)

    def build(vocab):
        enc = Encoder("CLIP-ViT-B/32")
        et = AttentiveEncoder(n_layers=3, feature_size=[7, 7, 768], heads=8, dropout=0.1)
        de = DecoderTransformer(decoder_type="transformer_decoder", embed_dim=768, vocab_size=len(vocab),
                                max_lengths=42, word_vocab=vocab, n_head=8, n_layers=1, dropout=0.1)
        return enc, et, de
    return build, preprocess


def tokenizer():
    import ast
    src = (SRC / "preprocess_data.py").read_text()
    keep = [n for n in ast.parse(src).body
            if (isinstance(n, ast.FunctionDef) and n.name in ("tokenize", "build_vocab", "encode"))
            or (isinstance(n, ast.Assign) and any(getattr(t, "id", "") == "SPECIAL_TOKENS" for t in n.targets))]
    ns = {}
    exec(compile(ast.Module(body=keep, type_ignores=[]), "preprocess_data", "exec"), ns)
    tok = lambda s: ns["tokenize"](s, add_start_token=True, add_end_token=True,  # noqa: E731
                                   punct_to_keep=[";", ","], punct_to_remove=["?", "."])
    return tok, ns["build_vocab"], ns["encode"]


def load_images(data, recs, preprocess):
    from PIL import Image
    a = torch.stack([preprocess(Image.open(Path(data) / "images" / r["filepath"] / "A" / r["filename"]).convert("RGB")) for r in recs])
    b = torch.stack([preprocess(Image.open(Path(data) / "images" / r["filepath"] / "B" / r["filename"]).convert("RGB")) for r in recs])
    return a, b


@torch.no_grad()
def greedy(models, a, b, device, idx2word, skip):
    enc, et, de = models
    for m in models:
        m.eval()
    out, times = [], []
    for i in range(len(a)):
        t = time.perf_counter()
        f1, f2 = enc(a[i:i + 1].to(device), b[i:i + 1].to(device))
        seq = de.sample(et(f1, f2), k=1)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t)
        out.append((" ".join(idx2word[w] for w in seq if w not in skip), len(seq) >= 43))
    return out, times


def bleu4(hyps, refs):
    from nltk.translate.bleu_score import corpus_bleu
    return corpus_bleu([[r.split() for r in rs] for rs in refs], [h.split() for h in hyps])


def cmd_train(a):
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    random.seed(a.seed)
    from torch.nn.utils.rnn import pack_padded_sequence
    caps = json.load(open(Path(a.data) / "LevirCCcaptions.json"))["images"]
    sel = select(caps)
    if a.limit:
        sel = {k: v[:a.limit] for k, v in sel.items()}
    tok, build_vocab, encode = tokenizer()
    ft_tokens = [(r["filename"], [tok(s["raw"]) for s in r["sentences"]]) for r in sel["finetune"]]
    vocab = build_vocab(ft_tokens, 5)  # train-subset words only (upstream: all splits)
    idx2word = {v: k for k, v in vocab.items()}
    skip = {vocab["<START>"], vocab["<END>"], vocab["<NULL>"]}

    pair_caps = []  # per pair: its (padded tokens, length) captions
    for _, seqs in ft_tokens:
        if NOCHANGE in seqs:  # upstream LEVIRCC.py collapses no-change pairs in train
            seqs = [NOCHANGE] * len(seqs)
        enc_seqs = [encode(s, vocab, allow_unk=True)[:42] for s in seqs]
        pair_caps.append([(e + [0] * (42 - len(e)), len(e)) for e in enc_seqs])
    n_samples = sum(len(c) for c in pair_caps)

    build, preprocess = load_upstream(device)
    imgA, imgB = load_images(a.data, sel["finetune"], preprocess)
    devA, devB = load_images(a.data, sel["development"], preprocess)
    dev_refs = [[" ".join(tok(s["raw"])[1:-1]) for s in r["sentences"]] for r in sel["development"]]
    enc, et, de = models = build(vocab)
    enc.fine_tune(True)
    for m in models:
        m.to(device)
    params = [p for m in models for p in m.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=a.lr)
    crit = torch.nn.CrossEntropyLoss()
    accum = a.accum

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    log, best, best_ep, start = [], -1.0, -1, 0
    if (out / "last.pth").exists():  # resume after a Colab disconnect
        st = torch.load(out / "last.pth", map_location="cpu", weights_only=False)
        for m, k in zip(models, ("encoder_dict", "encoder_trans_dict", "decoder_dict")):
            m.load_state_dict(st[k])
        opt.load_state_dict(st["opt"])
        log, best, best_ep, start = st["log"], st["best"], st["best_ep"], st["epoch"] + 1
        torch.set_rng_state(st["rng"])
        print(f"resumed at epoch {start}")
    gen = torch.Generator().manual_seed(a.seed)
    for _ in range(start):
        torch.randperm(len(pair_caps), generator=gen)
    t_start = time.time()

    for ep in range(start, a.epochs):
        for m in models:
            m.train()
        t0, tot, n = time.time(), 0.0, 0
        order = torch.randperm(len(pair_caps), generator=gen).tolist()
        n_steps = (len(order) + a.batch - 1) // a.batch
        opt.zero_grad()
        for step, i0 in enumerate(range(0, len(order), a.batch)):
            # Encode each pair once and decode all its captions against that encoding;
            # upstream re-encodes the pair for every caption (same loss, 5x the encoder work).
            pidx = order[i0:i0 + a.batch]
            caps_b = [c for j in pidx for c in pair_caps[j]]
            idx = torch.tensor(pidx)
            reps = torch.tensor([len(pair_caps[j]) for j in pidx], device=device)
            token = torch.tensor([c[0] for c in caps_b], device=device)
            tlen = torch.tensor([c[1] for c in caps_b], device=device)
            f1, f2 = enc(imgA[idx].to(device), imgB[idx].to(device))
            feat = et(f1, f2).repeat_interleave(reps, dim=0)
            scores, caps_sorted, dl, _ = de(feat, token, tlen)
            s = pack_padded_sequence(scores, dl, batch_first=True).data
            tg = pack_padded_sequence(caps_sorted[:, 1:], dl, batch_first=True).data
            loss = crit(s, tg.long())
            (loss / accum).backward()
            tot, n = tot + loss.item(), n + 1
            if (step + 1) % accum == 0:
                opt.step()
                opt.zero_grad()
            if step % 25 == 0:
                el = time.time() - t0
                print(f"epoch {ep} step {step + 1}/{n_steps} loss {loss.item():.3f} "
                      f"{el / (step + 1):.2f} s/step, epoch eta {el / (step + 1) * (n_steps - step - 1) / 60:.1f} min",
                      flush=True)
        hyps, _ = greedy(models, devA, devB, device, idx2word, skip)
        b4 = bleu4([h for h, _ in hyps], dev_refs)
        log.append({"epoch": ep, "train_loss": round(tot / max(n, 1), 4), "dev_bleu4": round(b4, 4),
                    "epoch_s": round(time.time() - t0, 1)})
        print(json.dumps(log[-1]), flush=True)
        state = {"encoder_dict": enc.state_dict(), "encoder_trans_dict": et.state_dict(),
                 "decoder_dict": de.state_dict(), "vocab": vocab}
        if b4 > best:
            best, best_ep = b4, ep
            torch.save(state, out / "best.pth")
        torch.save({**state, "opt": opt.state_dict(), "log": log, "best": best, "best_ep": best_ep,
                    "epoch": ep, "rng": torch.get_rng_state()}, out / "last.pth")
        if ep - best_ep >= a.patience:
            break
        if (time.time() - t_start) / 3600 + (log[-1]["epoch_s"] / 3600) > a.max_hours:
            print(f"stopping: next epoch would pass the {a.max_hours} h budget", flush=True)
            break
    json.dump({"log": log, "best_dev_bleu4": best, "best_epoch": best_ep, "vocab_size": len(vocab),
               "finetune_pairs": len(sel["finetune"]), "train_samples": n_samples,
               "config": {k: v for k, v in vars(a).items() if k != "func"}},
              open(out / "train_log.json", "w"), indent=1)
    print("best epoch", best_ep, "dev BLEU-4", round(best, 4))


def write_captions(path, run_id, model_id, rows, results, times):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for (role, r), (text, hit), t in zip(rows, results, times):
            f.write(json.dumps({"pair_id": f"{r['filepath']}/{r['filename'][:-4]}", "run_id": run_id,
                                "model": model_id, "condition": "B0", "poc_role": role, "raw_text": text,
                                "status": "ok", "output_tokens": len(text.split()), "hit_token_limit": hit,
                                "gen_s": round(t, 3)}) + "\n")


def roles(sel):
    rows = [("development_smoke" if i < 10 else "development", r) for i, r in enumerate(sel["development"])]
    return rows + [("screening", r) for r in sel["screening"]]


def cmd_caption(a):
    """Best retrained checkpoint on the same development + screening pairs as every B0 run."""
    device = torch.device(a.device)
    caps = json.load(open(Path(a.data) / "LevirCCcaptions.json"))["images"]
    rows = roles(select(caps))
    build, preprocess = load_upstream(device)
    st = torch.load(Path(a.out) / "best.pth", map_location="cpu", weights_only=False)
    vocab = st["vocab"]
    models = build(vocab)
    for m, k in zip(models, ("encoder_dict", "encoder_trans_dict", "decoder_dict")):
        m.load_state_dict(st[k], strict=True)
        m.to(device)
    A, B = load_images(a.data, [r for _, r in rows], preprocess)
    skip = {vocab["<START>"], vocab["<END>"], vocab["<NULL>"]}
    res, times = greedy(models, A, B, device, {v: k for k, v in vocab.items()}, skip)
    write_captions(Path(a.out) / "captions_retrained.jsonl", "B0_C02r20_rscama_retrained20_development_smoke+development+screening",
                   "C02-r20", rows, res, times)
    print("wrote", len(res))


def cmd_parity(a):
    """Released checkpoint, screening + development, with the shim or the real CUDA kernels."""
    device = torch.device(a.device)
    caps = json.load(open(Path(a.data) / "LevirCCcaptions.json"))["images"]
    rows = roles(select(caps))
    build, preprocess = load_upstream(device, shim=not a.kernels)
    vocab = build_author_vocab(Path(a.data) / "LevirCCcaptions.json", 5)
    st = torch.load(a.ckpt, map_location="cpu", weights_only=True)
    models = build(vocab)
    for m, k in zip(models, ("encoder_dict", "encoder_trans_dict", "decoder_dict")):
        m.load_state_dict(st[k], strict=True)
        m.to(device)
    A, B = load_images(a.data, [r for _, r in rows], preprocess)
    skip = {vocab["<START>"], vocab["<END>"], vocab["<NULL>"]}
    res, times = greedy(models, A, B, device, {v: k for k, v in vocab.items()}, skip)
    tag = "kernels" if a.kernels else "shim"
    write_captions(Path(a.out) / f"captions_released_{tag}_{device.type}.jsonl", f"parity_C02_{tag}_{device.type}",
                   "C02", rows, res, times)
    print("wrote", len(res))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="levir")
    ap.add_argument("--out", default="out")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "mps")
    sub = ap.add_subparsers(required=True)
    sub.add_parser("fetch").set_defaults(func=cmd_fetch)
    t = sub.add_parser("train")
    t.add_argument("--epochs", type=int, default=40)
    t.add_argument("--patience", type=int, default=8)
    t.add_argument("--batch", type=int, default=12, help="image pairs per step (x5 captions)")
    t.add_argument("--accum", type=int, default=1)
    t.add_argument("--max-hours", type=float, default=8.0)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--limit", type=int, default=0, help="smoke test only")
    t.set_defaults(func=cmd_train)
    sub.add_parser("caption").set_defaults(func=cmd_caption)
    p = sub.add_parser("parity")
    p.add_argument("--ckpt", default="RSCaMa.pth")
    p.add_argument("--kernels", action="store_true")
    p.set_defaults(func=cmd_parity)
    a = ap.parse_args()
    a.func(a)


if __name__ == "__main__":
    main()


## 1. Fetch only the selected pairs (hashes are checked against the frozen manifest)

In [ ]:
!cd /content && python b/rscama_colab.py --data /content/levir --out {OUT} fetch
assert _exit_code == 0, 'step failed: see the traceback above'

## 2. Parity: released checkpoint on CUDA with the same pure-PyTorch shim used on the Mac

In [ ]:
from huggingface_hub import hf_hub_download
ck = hf_hub_download('lcybuaa/RSCaMa', 'RSCaMa.pth', revision='3e527e6656b62cc615eab1d2f3fb73777cbbf62c',
                     local_dir='/content/weights', token=False)
os.environ['CKPT'] = ck
!cd /content && python b/rscama_colab.py --data /content/levir --out {OUT} parity --ckpt {ck}
assert _exit_code == 0, 'step failed: see the traceback above'

## 3. Train (resumes from `last.pth` if present)
Each step takes 12 image pairs and decodes all 5 captions of each against one encoding (60 captions per update, upstream uses 64). Prints progress every 25 steps; stops after 8 h so captioning fits in Kaggle's 12 h limit.

In [ ]:
!cd /content && python b/rscama_colab.py --data /content/levir --out {OUT} train --epochs 40 --patience 8 --batch 12 --max-hours 8
assert _exit_code == 0, 'step failed: see the traceback above'

## 4. Caption the development + screening pairs with the best checkpoint

In [ ]:
!cd /content && python b/rscama_colab.py --data /content/levir --out {OUT} caption
assert _exit_code == 0, 'step failed: see the traceback above'

## 5. Optional: parity with the real Mamba CUDA kernels
Building the kernels can take 10-30 minutes; skip this cell if it fails, the results above are complete.

In [ ]:
%%bash
pip -q install --no-build-isolation causal-conv1d mamba-ssm && \
cd /content && python b/rscama_colab.py --data /content/levir --out $OUT parity --ckpt $CKPT --kernels \
|| echo "kernel parity skipped" 

## 6. Package results (captions + logs only)

In [ ]:
import glob, hashlib, json, platform, shutil, torch, transformers, zipfile
env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda,
       'transformers': transformers.__version__, 'python': platform.python_version(),
       'best_pth_sha256': hashlib.sha256(open(f'{OUT}/best.pth', 'rb').read()).hexdigest()}
json.dump(env, open(f'{OUT}/colab_env.json', 'w'), indent=1)
shutil.copy('/content/levir/fetch.json', f'{OUT}/fetch.json')
ZIP = '/kaggle/working/b_rscama_results.zip' if ON_KAGGLE else '/content/b_rscama_results.zip'
with zipfile.ZipFile(ZIP, 'w') as z:
    for p in glob.glob(f'{OUT}/*.json') + glob.glob(f'{OUT}/*.jsonl'):
        z.write(p, os.path.basename(p))
print(env, ZIP)
if ON_COLAB:
    from google.colab import files
    files.download(ZIP)
else:
    print('Download', ZIP, 'from the Output panel (Kaggle: right-hand sidebar > Output).')